#### 연습 문제
- Doc2Vec을 이용하여 감성 분석
- ratings_train.txt 파일 로드
    - 특수 문자, 2칸 이상의 공백을 제거하는 정규화 함수 이용
    - document column의 중복 제거
    - 빈 텍스트가 존재한다면 제거
    - 상위 5000개 데이터를 학습 데이터로
- 토큰화 Komoran
    - 품사 필터: NNP, NNG, VV, VA, MAG, XR 사용
    - 불용어: 하다, 되다, 이다, 것, 수, 거
- 독립변수(document), 종속변수(label) 데이터를 나눠주고 train, text로 데이터 분할(8:2)
- Doc2Vec 객체 생성하여 벡터화
    - 매개변수
        - vector_size = 200
        - window = 5
        - min_count = 2
        - dm = 1
        - negative = 5
        - seed = 42
        - epochs = 50
    - train data 사용하여 학습
- Doc2Vec 객체에서 train, test 데이터를 infer_vector() 함수를 이용하여 벡터 데이터 생성
- ML 분류 모델을 이용하여 임베딩된 데이터를 독립 변수로 사용하여 학습
    - 정확도 확인
    - Logistic
        - max_iter = 2000
        - random_state = 42
    - LinearSVC
        - random_state = 42
- test 데이터를 이용하여 2개의 모델 중 정확도 높은 모델을 검색

In [1]:
from konlpy.tag import Komoran
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

import numpy as np
import pandas as pd
import re

##### **<font color = red>우당탕탕 내 풀이ㅠ</font>**

In [2]:
df = pd.read_csv('../data/ratings_train.txt', sep = '\t')

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   id        150000 non-null  int64
 1   document  149995 non-null  str  
 2   label     150000 non-null  int64
dtypes: int64(2), str(1)
memory usage: 3.4 MB


In [4]:
df.drop('id', axis = 1, inplace = True)
df.dropna(inplace = True)
df.reset_index(drop = True, inplace = True)

In [5]:
for idx, data in enumerate(df['document']):
    df.loc[idx, 'document'] = data.strip()

In [6]:
df.drop_duplicates('document', inplace=True)
df.reset_index(drop = True, inplace = True)

In [7]:
df['document'].value_counts()

document
아 더빙.. 진짜 짜증나네요 목소리                                              1
흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나                                1
너무재밓었다그래서보는것을추천한다                                                1
교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정                                    1
사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 던스트가 너무나도 이뻐보였다    1
                                                                ..
인간이 문제지.. 소는 뭔죄인가..                                              1
평점이 너무 낮아서...                                                    1
이게 뭐요? 한국인은 거들먹거리고 필리핀 혼혈은 착하다?                                  1
청춘 영화의 최고봉.방황과 우울했던 날들의 자화상                                      1
한국 영화 최초로 수간하는 내용이 담긴 영화                                         1
Name: count, Length: 146182, dtype: int64

In [ ]:
def normalize(text):
    # 특수 문자, 공백에 대한 처리
    text = re.sub(r'[^가-힣0-9a-zA-Z\s\.]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [ ]:
for idx, data in enumerate(df['document']):
    df.loc[idx, 'document'] = normalize(str(data))

In [10]:
flag = (df['document'] == '')
df.loc[flag, ]

,document,label
972,,0
1834,,1
2151,,1
2495,,0
2639,,1
...,...,...
142828,,0
143383,,0
145577,,1
145610,,0


In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 146182 entries, 0 to 146181
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   document  146182 non-null  str  
 1   label     146182 non-null  int64
dtypes: int64(1), str(1)
memory usage: 2.2 MB


In [12]:
komoran = Komoran()

In [13]:
allow_pos = ['NNP', 'NNG', 'VV', 'VA', 'MAG', 'XR']
stop_word = ['하다', '되다', '이다', '것', '수', '거']

In [14]:
def tokenize(text):

    tokens = []
    for word, pos in komoran.pos(text):
        if pos in allow_pos:
            if pos in ['VV', 'VA']:
                word += '다'
            if word not in stop_word and len(word) > 1:
                tokens.append(word)
    return tokens

In [15]:
X = df['document'][:5000].values
y = df['label'][:5000].values

In [16]:
X

<StringArray>
[                                                                   '아 더빙.. 진짜 짜증나네요 목소리',
                                                      '흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나',
                                                                      '너무재밓었다그래서보는것을추천한다',
                                                          '교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정',
                          '사이몬페그의 익살스런 연기가 돋보였던 영화 스파이더맨에서 늙어보이기만 했던 커스틴 던스트가 너무나도 이뻐보였다',
                                            '막 걸음마 뗀 3세부터 초등학교 1학년생인 8살용영화. ...별반개도 아까움.',
                                                                  '원작의 긴장감을 제대로 살려내지못했다.',
 '별 반개도 아깝다 욕나온다 이응경 길용우 연기생활이몇년인지..정말 발로해도 그것보단 낫겟다 납치.감금만반복반복..이드라마는 가족도없다 연기못하는사람만모엿네',
                                                                 '액션이 없는데도 재미 있는 몇안되는 영화',
                                            '왜케 평점이 낮은건데 꽤 볼만한데.. 헐리우드식 화려함에만 너무 길들여져 있나',
 ...
                                                                       

In [17]:
X = [tokenize(doc) for doc in X]

In [18]:
X

[['더빙', '진짜', '짜증', '나다', '목소리'],
 ['포스터', '초딩', '영화', '오버', '연기', '가볍다'],
 [],
 ['교도소', '이야기', '솔직히', '재미', '없다', '평점', '조정'],
 ['익살', '연기', '돋보이다', '영화', '스파이더맨', '늙다', '보이다', '커스틴 던스트', '너무나'],
 ['걸음마', '떼다', '초등학교', '학년', '영화', '반개', '아깝다'],
 ['원작', '긴장감', '제대로', '살리다'],
 ['반개',
  '아깝다',
  '나오다',
  '이응경',
  '길용우',
  '연기',
  '생활',
  '정말',
  '발로',
  '납치',
  '감금',
  '반복',
  '반복',
  '드라마',
  '가족',
  '없다',
  '연기',
  '못하다',
  '사람',
  '모이다'],
 ['액션', '없다', '재미', '있다', '영화'],
 ['평점', '낮다', '보다', '헐리우드', '화려', '너무', '길들이다', '있다'],
 [],
 ['눈물', '나서다', '죽다', '향수', '자극', '허진호', '감성', '절제', '멜로', '달인'],
 ['울다', '손들다', '횡단보도', '건너다', '뛰쳐나오다', '이범수', '연기', '드럽다'],
 ['담백', '깔끔', '좋다', '신문', '기사', '로만', '보다', '보다', '자꾸', '잊어버리다', '사람'],
 ['취향',
  '존중',
  '진짜',
  '극장',
  '보다',
  '영화',
  '가장',
  '재다',
  '감동',
  '스토리',
  '어거지',
  '감동',
  '어거지'],
 ['매번', '긴장'],
 ['사람',
  '웃기다',
  '바스코',
  '이기다',
  '락스',
  '바비',
  '이기다',
  '아이돌',
  '깔다',
  '그냥',
  '까다',
  '안달',
  '보이다'],
 ['굿바이 레닌', '표절', '이해', '갈수록', '

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, stratify = y, random_state = 42
)

In [20]:
# doc2vec은 문장 하나씩 ID를 붙여준다.
tagged = []
for idx, token in enumerate(X_train):
    # idx: 위치
    # token: 리스트의 원소
    tagged.append(
        TaggedDocument(words = token, tags = [f'DOC_{idx}'])
    )

tagged

[TaggedDocument(words=['아하'], tags=['DOC_0']),
 TaggedDocument(words=['개도', '아깝다'], tags=['DOC_1']),
 TaggedDocument(words=['어떻다', '평점', '낮다', '있다'], tags=['DOC_2']),
 TaggedDocument(words=['이영화', '논하다'], tags=['DOC_3']),
 TaggedDocument(words=['죄다', '없다'], tags=['DOC_4']),
 TaggedDocument(words=['가장', '어깨', '짊어지다', '책임감', '가장', '리얼', '표현', '영화', '여담', '이후', '조 루이스', '이전', '복싱', '황금기', '너무나', '표현', '진짜', '복싱', '춘추', '전국 시대'], tags=['DOC_5']),
 TaggedDocument(words=['린즈링', '너무', '이상', '영화', '쓰레기'], tags=['DOC_6']),
 TaggedDocument(words=['여자친구', '전쟁', '치루다'], tags=['DOC_7']),
 TaggedDocument(words=['솔직히', '이영화', '재미없다', '평점', '세상', '재미있다', '영화', '하나', '없다', '참고', '기똥차다', '즐겁다', '보다', '다음', '시리즈'], tags=['DOC_8']),
 TaggedDocument(words=['각본', '캐스팅', '전부', '완벽', '99년', '허준', '김재철', '전사', '장이', '발치', '리메이크', '해서', '망하다', '이제', '아무도', '기억', '드라마'], tags=['DOC_9']),
 TaggedDocument(words=['전쟁', '이후', '아픔', '그린', '영화', '마지막', '장면', '압권', '유대인', '알다', '더럽다', '도움', '필요', '하니', '동행', '토마스', '더럽

In [21]:
len(tagged)

4000

In [22]:
model = Doc2Vec(
    documents = tagged,
    vector_size = 200,
    window = 5,
    min_count = 2,
    dm = 1,
    epochs = 50,
    seed = 42,
    negative = 5
)

In [23]:
X_train_vec = [model.infer_vector(data) for data in X_train]

In [24]:
X_test_vec = [model.infer_vector(data) for data in X_test]

In [25]:
logi = LogisticRegression(random_state = 42, max_iter = 2000)
svc = LinearSVC(random_state = 42)

In [26]:
logi.fit(X_train_vec, y_train)

,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",2000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lb

In [27]:
pred_logi = logi.predict(X_test_vec)

In [28]:
svc.fit(X_train_vec, y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo random number generation for shuffling the data forthe dual coordinate descent (if ``dual=True``). When ``dual=False`` theunderlying implementation of :class:`LinearSVC` is not random and``random_state`` has no effect on the results.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_scaling`. This scaling allows the intercept term to have adifferent regularization behavior compared to the other features.",1
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adj

In [29]:
pred_svc = svc.predict(X_test_vec)

In [30]:
print('LogisticRegression')
print(classification_report(y_test, pred_logi))
print()
print('LinearSVC')
print(classification_report(y_test, pred_svc))

LogisticRegression
              precision    recall  f1-score   support

           0       0.74      0.72      0.73       500
           1       0.73      0.75      0.74       500

    accuracy                           0.74      1000
   macro avg       0.74      0.74      0.74      1000
weighted avg       0.74      0.74      0.74      1000


LinearSVC
              precision    recall  f1-score   support

           0       0.74      0.74      0.74       500
           1       0.74      0.74      0.74       500

    accuracy                           0.74      1000
   macro avg       0.74      0.74      0.74      1000
weighted avg       0.74      0.74      0.74      1000



##### **강사님 풀이**

In [31]:
df = pd.read_csv('../data/ratings_train.txt', sep = '\t')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   id        150000 non-null  int64
 1   document  149995 non-null  str  
 2   label     150000 non-null  int64
dtypes: int64(2), str(1)
memory usage: 3.4 MB


In [32]:
df.dropna(inplace = True)

In [ ]:
# 텍스트 정규화 함수
# 나는 중복제거한 이후에 nomalize를 했고,
# 강사님은 nomalize를 하고 중복 제거를 하셨다.
    # 이랬더니 개수 차이가 400개? 가까이 났음... 성능은 1% 정도 차이... 오히려 내려감

def normalize(text):
    text = re.sub(r'[^가-힣0-9a-zA-Z\s\.]', ' ', str(text))     # 달라진 부분
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [ ]:
df['document'] = df['document'].map(normalize)

In [35]:
df = df.loc[
    ~(df['document'] == ''),
]

In [36]:
df.info()

<class 'pandas.DataFrame'>
Index: 149559 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   id        149559 non-null  int64
 1   document  149559 non-null  str  
 2   label     149559 non-null  int64
dtypes: int64(2), str(1)
memory usage: 4.6 MB


In [37]:
df.drop_duplicates('document', inplace = True)

In [38]:
allow_pos = ['NNP', 'NNG', 'VV', 'VA', 'MAG', 'XG']
stop_word = ['하다', '되다', '이다', '것', '수', '거']

In [39]:
komoran = Komoran()

In [40]:
def tokenize(text):
    tokens = []
    for word, pos in komoran.pos(text):
        if pos in allow_pos and word not in stop_word:
            tokens.append(word)
    return tokens

In [41]:
df3 = df.head(5000)

In [43]:
tokenize_sentence = [
    tokenize(text) for text in df3['document'].values
]

In [44]:
X = tokenize_sentence
y = df3['label'].values

In [45]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, stratify = y, random_state = 42
)

In [46]:
def tagged_docs(token_data):
    tagged = []
    for idx, toks in enumerate(token_data):
        # toks의 길이가 0이라면 학습에서 큰 의미가 없음 → 제외
        if len(toks) == 0:
            continue

        tagged.append(
            TaggedDocument(
                words = toks, tags = [f'DOC_{idx}']
            )
        )
    
    return tagged

In [47]:
X_train_tag = tagged_docs(X_train)

In [48]:
print(len(X_train_tag), len(X_train))

3920 4000


In [49]:
# Doc2Vec 객체 생성
model = Doc2Vec(
    documents = X_train_tag,
    vector_size = 200,
    window = 5,
    min_count = 2,
    negative = 5,
    seed = 42,
    epochs = 50
)

In [51]:
model2 = Doc2Vec(
    vector_size = 200,
    window = 5,
    min_count = 2,
    negative = 5,
    seed = 42,
    epochs = 50 
)

model2.build_vocab(X_train_tag)

model2.train(
    X_train_tag, total_examples = len(X_train_tag), epochs = 50
)

In [52]:
print(len(model.wv))
print(len(model2.wv))

2750
2750


In [53]:
print(len(model.dv))

3920


In [54]:
def infer_vector(model, norm_tokens, epochs = 50):
    # model: 임베딩 모델
    # norm_texts: 텍스트 정규화가 끝나고 토큰화가 완료된 데이터

    result = []

    for tokens in norm_tokens:
        if len(tokens) == 0:
            result.append(
                np.zeros(model.vector_size, dtype = np.float32)
            )
        else:
            vec = model.infer_vector(tokens, epochs = epochs)
            result.append(vec)
    
    return np.array(result)

In [55]:
X_train_vec = infer_vector(model, X_train)
X_test_vec = infer_vector(model, X_test)

In [56]:
X_train_vec.shape

(4000, 200)

In [57]:
def eval_clf(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print(classification_report(y_test, pred))

In [58]:
logi = LogisticRegression(max_iter = 2000, random_state = 42)
svc = LinearSVC(random_state = 42)

In [59]:
eval_clf(logi, X_train_vec, X_test_vec, y_train, y_test)
eval_clf(svc, X_train_vec, X_test_vec, y_train, y_test)

              precision    recall  f1-score   support

           0       0.71      0.74      0.73       501
           1       0.73      0.70      0.71       499

    accuracy                           0.72      1000
   macro avg       0.72      0.72      0.72      1000
weighted avg       0.72      0.72      0.72      1000

              precision    recall  f1-score   support

           0       0.71      0.76      0.73       501
           1       0.74      0.70      0.72       499

    accuracy                           0.73      1000
   macro avg       0.73      0.73      0.73      1000
weighted avg       0.73      0.73      0.73      1000



In [60]:
df

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화 스파이더맨에서 늙어보이기만 했던 커스틴 ...,1
...,...,...,...
149995,6222902,인간이 문제지.. 소는 뭔죄인가..,0
149996,8549745,평점이 너무 낮아서...,1
149997,9311800,이게 뭐요 한국인은 거들먹거리고 필리핀 혼혈은 착하다,0
149998,2376369,청춘 영화의 최고봉.방황과 우울했던 날들의 자화상,1


In [61]:
# DataFrame을 train, test 분할

train_df, test_df = train_test_split(
    df, test_size = 0.2, random_state = 42, stratify = df['label']
)

In [62]:
train_df['label'].value_counts()

label
0    58271
1    57515
Name: count, dtype: int64

In [63]:
test_df['label'].value_counts()

label
0    14568
1    14379
Name: count, dtype: int64